In [28]:
# ==========================================
# BLOQUE 1: Cargar Excel, Filtrar Estación CEN y Exportar
# ==========================================
import pandas as pd
import numpy as np
import unicodedata
from google.colab import files

# 1. Configuración
NOMBRE_ARCHIVO_ENTRADA = "BD_2015.xlsx"
ESTACION_OBJETIVO = "CEN"
AÑO_DATOS = 2015

# Función para normalizar nombres de columnas:
# quita espacios, mayúsculas y acentos
def norm(s):
    s = str(s).strip().lower()
    s = ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )
    return s

# 2. Cargar Excel sin borrar vacíos
df_raw = pd.read_excel(NOMBRE_ARCHIVO_ENTRADA, sheet_name=0)
df_raw.columns = [str(c).strip() for c in df_raw.columns]

# 3. Detectar columna de estación
col_estacion = next(
    (c for c in df_raw.columns if norm(c) in ("estacion", "est", "station")),
    None
)

# 4. Filtrar CEN
if col_estacion is not None:
    # Formato largo: hay columna Estacion
    df_cen = df_raw[
        df_raw[col_estacion].astype(str).str.strip().str.upper()
        == ESTACION_OBJETIVO.upper()
    ].copy()
else:
    # Formato ancho: columnas tipo CEN_O3, CEN_PM10, etc.
    cols_cen = []
    for c in df_raw.columns:
        nc = norm(c)
        if nc in ("fecha", "hora", "mes", "dia") or nc.startswith(norm(ESTACION_OBJETIVO)):
            cols_cen.append(c)
    df_cen = df_raw[cols_cen].copy()

# 5. Exportar Excel conservando NaN / huecos
NOMBRE_EXCEL_SALIDA = f"BD_{AÑO_DATOS}_{ESTACION_OBJETIVO}.xlsx"
df_cen.to_excel(NOMBRE_EXCEL_SALIDA, index=False)

print(f"Excel filtrado guardado como: {NOMBRE_EXCEL_SALIDA}")
print(f"Registros para {ESTACION_OBJETIVO}: {len(df_cen)}")
print("Columnas:", list(df_cen.columns))

Excel filtrado guardado como: BD_2015_CEN.xlsx
Registros para CEN: 8760
Columnas: ['STATION', 'DATE', 'HOUR', 'O3', 'NO', 'NO2', 'NOX', 'SO2', 'CO', 'PM10', 'PM2.5', 'IT', 'ET', 'RH', 'WS', 'WD', 'PP', 'ATM', 'RS', 'UVI']


In [29]:
# ==========================================
# BLOQUE 2: Generar archivo JS datos_año.js
# ==========================================
import json

# Columnas exactas que debe tener la matriz "h", en este orden
cols_parametros = [
    "O3", "CO", "NO2", "SO2", "PM10", "PM2.5",
    "PB", "TMP", "RH", "WDS", "WSP", "MES", "DIA"
]

# Alias para detectar columnas con nombres distintos en el Excel
alias = {
    "O3": ["o3", "ozono"],
    "CO": ["co"],
    "NO2": ["no2"],
    "SO2": ["so2"],
    "PM10": ["pm10"],
    "PM2.5": ["pm2.5", "pm25", "pm2_5", "pm2·5"],
    "PB": ["pb", "plomo"],
    "TMP": ["tmp", "temp", "temperatura"],
    "RH": ["rh", "hr", "hum", "humedad"],
    "WDS": ["wds", "wd", "dir_viento", "direccion_viento"],
    "WSP": ["wsp", "ws", "vel_viento", "velocidad_viento"],
    "MES": ["mes", "month"],
    "DIA": ["dia", "día", "day"],
}

def norm(s):
    s = str(s).strip().lower()
    s = ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )
    return s

def buscar_columna(df, key):
    # 1. coincidencia exacta normalizada
    nk = norm(key)
    for c in df.columns:
        if norm(c) == nk:
            return c

    # 2. alias
    for a in alias.get(key, []):
        na = norm(a)
        for c in df.columns:
            if norm(c) == na:
                return c

    return None

# Construir matriz con exactamente las 13 columnas, en orden.
# Si falta una columna, se rellena con NaN para no romper el formato.
df_matriz = pd.DataFrame(index=df_cen.index)

for key in cols_parametros:
    src = buscar_columna(df_cen, key)
    if src is not None:
        df_matriz[key] = df_cen[src]
    else:
        df_matriz[key] = np.nan

# Convertir NaN a None para que json.dumps escriba null
df_matriz = df_matriz.astype(object).where(pd.notna(df_matriz), None)

matriz_h = df_matriz.values.tolist()

# Estructura JS
js_obj = {
    "year": AÑO_DATOS,
    "h": matriz_h
}

NOMBRE_JS_SALIDA = f"datos_{AÑO_DATOS}.js"

with open(NOMBRE_JS_SALIDA, "w", encoding="utf-8") as f:
    f.write(
        f"window.DATA_{AÑO_DATOS} = "
        + json.dumps(js_obj, ensure_ascii=False, allow_nan=False, separators=(",", ":"))
        + ";"
    )

print(f"JS generado: {NOMBRE_JS_SALIDA}")
print(f"Filas en h: {len(matriz_h)}")
print(f"Columnas en h: {len(cols_parametros)}")

JS generado: datos_2015.js
Filas en h: 8760
Columnas en h: 13


In [30]:
files.download(NOMBRE_EXCEL_SALIDA)
files.download(NOMBRE_JS_SALIDA)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>